# 3.2 — MLP baseline 5-fold trên folds chung

Cùng cấu hình MLP đã chọn ở DL task 2.1/2.5 (StandardScaler 19 cột số/boolean + one-hot 5 category → 128 → 64 → 1,
ReLU, dropout 0.20/0.10, AdamW lr 1e-3, wd 1e-4, batch 4096, tối đa 12 epoch, patience 3 theo AUC, clip 1.0, BCE không trọng số),
nhưng chạy trên 5 fold chung. Preprocessor fit riêng trong từng fold. Test = trung bình xác suất 5 fold.

**Input:** competition + `ccf-shared-code` + output notebook 3.0. **Accelerator:** GPU T4.
**Kiểm tra:** OOF AUC ≈ 0.944–0.945 (2.1 holdout: 0.94462).

In [ ]:
import sys
from pathlib import Path

# shared/ code: local repo or the Kaggle dataset that holds features.py + dl.py
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("shared code:", feat.parent)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

tr, te, y, train_id, test_id = F.load()
fold = F.load_folds(train_id)          # from notebook 3.0 output (folds_5_seed9.csv)
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST and set(np.unique(fold)) == set(range(F.N_FOLDS))
log(f"train {tr.shape}, test {te.shape}, fraud rate {y.mean():.4%}")

In [ ]:
import torch
import dl as D
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NAME = "mlp_base"
CFG = D.TrainConfig(lr=1e-3, weight_decay=1e-4, batch_size=4096, max_epochs=12, patience=3, grad_clip=1.0)
print("Device:", D.DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

NUM = F.NUMS + F.BOOLS

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), F.CATS),
    ])

In [ ]:
oof = np.zeros(len(tr))
test_pred = np.zeros(len(te))
fold_info = []
for f, a, b in F.iter_folds(fold):
    pre = make_preprocessor().fit(tr.iloc[a])
    Xa = pre.transform(tr.iloc[a]).astype(np.float32)
    Xb = pre.transform(tr.iloc[b]).astype(np.float32)
    Xt = pre.transform(te).astype(np.float32)
    model = D.MLP(Xa.shape[1])
    model, o, info = D.train_fold(model, (Xa,), y[a], (Xb,), y[b], CFG, seed=F.SEED + f, log_prefix=f"fold {f} ")
    oof[b] = o
    test_pred += D.predict(model, (Xt,)) / F.N_FOLDS
    fold_info.append({"fold": f, "best_epoch": info["best_epoch"], "auc": float(info["best_auc"])})
    log(f"fold {f}: AUC={info['best_auc']:.5f} best_epoch={info['best_epoch']} (in_dim={Xa.shape[1]})")

m = F.save_predictions(NAME, train_id, oof, test_id, test_pred, y, fold,
                       extra={"source": "3.2 MLP baseline (DL 2.1 config)", "config": CFG.__dict__, "folds": fold_info})
print(json.dumps({k: m[k] for k in ["oof_roc_auc", "oof_average_precision", "fold_roc_auc_mean", "fold_roc_auc_std"]}, indent=2))
log("done")